In [9]:
import os
import json
import torch

In [2]:
path = "../bert_data/topic_id-p2_5"

test_files = []
for file in os.listdir(path):
    if "bert.pt" in file and "test" in file:
        test_files.append(path + "/" + file)

test_files = sorted(test_files)

In [11]:
def save_data(path, filename, data, length):
    save_file = f"{path}-{length}/{filename}"
    torch.save(data, save_file)
    
    json_file = save_file.replace('bert.pt', 'json')
    with open(json_file, 'w') as save:
        save.write(json.dumps(data))

for file in test_files:
    data_200 = []
    data_400 = []
    data_600 = []
    data_800 = []
    data_1000 = []
    data_1200 = []
    data_1400 = []
    data_1600 = []
    data_1800 = []
    data_2000 = []
    print(f"Loading {file}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src_len = len(data['src'])
        if src_len <= 200:
            data_200.append(data)        
        elif src_len <= 400:
            data_400.append(data)
        elif src_len <= 600:
            data_600.append(data)        
        elif src_len <= 800:
            data_800.append(data)
        elif src_len <= 1000:
            data_1000.append(data)        
        elif src_len <= 1200:
            data_1200.append(data)
        elif src_len <= 1400:
            data_1400.append(data)        
        elif src_len <= 1600:
            data_1600.append(data)
        elif src_len <= 1800:
            data_1800.append(data)        
        elif src_len <= 2000:
            data_2000.append(data)
            
    filename = file.split("/")[-1]
    
    save_data(path, filename, data_200, 200)
    save_data(path, filename, data_400, 400)
    save_data(path, filename, data_600, 600)
    save_data(path, filename, data_800, 800)
    save_data(path, filename, data_1000, 1000)
    save_data(path, filename, data_1200, 1200)
    save_data(path, filename, data_1400, 1400)
    save_data(path, filename, data_1600, 1600)
    save_data(path, filename, data_1800, 1800)
    save_data(path, filename, data_2000, 2000)

Loading ../bert_data/topic_id-p2_5/xlsum.test.0.bert.pt...
Loading ../bert_data/topic_id-p2_5/xlsum.test.1.bert.pt...
Loading ../bert_data/topic_id-p2_5/xlsum.test.2.bert.pt...


In [23]:
import gc
import os
import glob
import json
import re
import shutil
from os.path import join as pjoin

import torch
from multiprocess import Pool

def get_long_data(raw_path_1):
    
    lang_1 = raw_path_1.split("/")[-1]
    
    save_path_1 = f"{raw_path_1}_long"
    if os.path.exists(save_path_1):
        shutil.rmtree(save_path_1)
    os.mkdir(save_path_1)
    
    datasets = ['train', 'valid', 'test']
    
    long_data_idx = {}
    for corpus_type in datasets:
        print(f"Looping through {raw_path_1} {corpus_type} dataset...")
        
        # Loop through the reference dataset
        for json_f in glob.glob(pjoin(raw_path_1, '*' + corpus_type + '.*.json')):
            dataset_name = json_f.split('/')[-1].split(".")[0]  # --> xlsum
            idx = json_f.split('/')[-1].split(".")[-2] # --> index: 1-19 (train), 1-3 (valid, test)
            filename = f"/{dataset_name}.{corpus_type}.{idx}.bert.pt"
            
            save_file = save_path_1 + filename
            
            with open(json_f) as json_f:
                data = json.load(json_f)
                datasets = []
    
                # Get data with total tokens > 512
                for i in range(len(data)):
                    src = data[i]['src']
                    if len(src) > 512:
                        datasets.append(data[i])
    
                        # Save the index (key) for data with long tokens
                        if f"{corpus_type}_{idx}" in long_data_idx.keys():
                            long_data_idx[f"{corpus_type}_{idx}"].append(i) 
                        else:
                            long_data_idx[f"{corpus_type}_{idx}"] = [i]
                        
                print(f"Saving {len(datasets)} instances to {save_file}...")
                torch.save(datasets, save_file)
            
                save_file_json = save_file.replace('bert.pt', 'json')
                with open(save_file_json, 'w') as f:
                    f.write(json.dumps(datasets))

In [25]:
get_long_data("../bert_data/topic_id-p2_5")

Looping through ../bert_data/topic_id-p2_5 train dataset...
Saving 620 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.8.bert.pt...
Saving 619 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.11.bert.pt...
Saving 623 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.9.bert.pt...
Saving 636 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.12.bert.pt...
Saving 839 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.2.bert.pt...
Saving 114 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.19.bert.pt...
Saving 637 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.14.bert.pt...
Saving 607 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.10.bert.pt...
Saving 613 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.4.bert.pt...
Saving 632 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.18.bert.pt...
Saving 654 instances to ../bert_data/topic_id-p2_5_long/xlsum.train.17.bert.pt...
Saving 1303 instances to ../bert_data/topi